[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/11_sliding_window.ipynb)

# 🔴 Hard: Sliding Window Attention

Implement **Sliding Window Attention** — used in Longformer, Mistral, etc. for efficient long-context processing.

Each position $i$ can only attend to positions $j$ where $|i - j| \le w$ (the window size).

### Signature
```python
def sliding_window_attention(Q, K, V, window_size):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
    # window_size: int — position i attends to [i-w, i+w]
```

### Rules
- Do **NOT** use sparse attention libraries
- Mask positions outside the window with `-inf`
- `window_size=0`: only self — output should equal V
- `window_size >= seq_len`: equivalent to full attention

In [2]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.0 MB/s eta 0:00:00


In [1]:
import torch
import math

In [6]:
# ✏️ YOUR IMPLEMENTATION HERE

# NOTE: while this behaves in the O(S*W) in logic, this method still
# calculates attention over the entire context, so in practice, this is still O(S^2)

def sliding_window_attention(Q, K, V, window_size):

  # Cureent QKV shape (B, S, D)
  B, S, D = Q.shape

  # (B, S, D) @ (B, D, S) -> (B, S, S)
  scores = torch.matmul(Q, K.transpose(-1,-2)) / math.sqrt(D)

  # Create the sliding window mask
  indices = torch.arange(S) # [0,1,2,3, ...]

  # calculate the |i - j| for every pair
  # BROADCASTING: (S, 1) - (1, S) creates a (S, S) matrix of differences
  distance = torch.abs(indices.view(S, 1) - indices.view(1,S))

  # mask is true (hide) when the distance is larger than the window size
  mask = distance > window_size

  # Apply the mask to scores
  scores = scores.masked_fill(mask, float('-inf'))

  # softmax over masked scores to remove tokens outside of the window_size
  weights = torch.softmax(scores, dim=-1)

  x = torch.matmul(weights, V)

  return x


In [7]:
# 🧪 Debug
Q = torch.randn(1, 6, 8)
K = torch.randn(1, 6, 8)
V = torch.randn(1, 6, 8)

out = sliding_window_attention(Q, K, V, window_size=1)
print("Output shape:", out.shape)  # (1, 6, 8)

# window=0 should return V
out0 = sliding_window_attention(Q, K, V, window_size=0)
print("window=0 == V?", torch.allclose(out0, V, atol=1e-5))

Output shape: torch.Size([1, 6, 8])
window=0 == V? True


In [8]:
from torch_judge import check
check('sliding_window')


🧪 Testing: Sliding Window Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (1.7ms)
  ✅ [2/5] window_size=0 — only sees itself (0.7ms)
  ✅ [3/5] Large window equals full attention (2.3ms)
  ✅ [4/5] Distant tokens don't affect output (1.9ms)
  ✅ [5/5] Gradient flow (0.8ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (7.3ms total)
  Progress saved. Run status() to see your dashboard.

